In [ ]:
import os
from dotenv import load_dotenv
import torch
import numpy as np
from tqdm.notebook import tqdm
import pandas as pd
from src.utils import split_dataframe
from transformers import AutoTokenizer, AutoModelForSequenceClassification, DataCollatorWithPadding, TrainingArguments, Trainer, EarlyStoppingCallback 
from datasets import Dataset
from sklearn.metrics import f1_score, accuracy_score, hamming_loss, roc_auc_score, precision_score, recall_score, classification_report
from scipy.special import expit
import torch.nn as nn
import random
from pathlib import Path
import json
import joblib

tqdm.pandas()
load_dotenv()

local_model_path = "../models/base/ModernBERT-base"

### Determinisztikus környezet beállítása (Set deterministic environment)

In [ ]:
SEED = int(os.getenv("SEED", "42"))

os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
torch.cuda.empty_cache()
torch.backends.cudnn.deterministic = True
torch.use_deterministic_algorithms(True)
torch.backends.cudnn.benchmark = False

np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)
random.seed(SEED)

#### A rendelkezésre álló legjobb eszköz kiválasztása (Select the best available device)

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

if device.type == 'cuda':
    print(f"Current device: {torch.cuda.get_device_name(0)}")
else:
    print("Current device: CPU")

## ModernBERT finomhangolása gyakori ICD-10-CM chapter osztályozásra (Fine-tuning ModernBERT for frequent ICD-10-CM chapter classification)

In [ ]:
processed_data_dir = Path("../data/processed")
extension = Path(".parquet")

train_data_dirs = [
    processed_data_dir / "frequent_chapter/without_dropped_sections",
    processed_data_dir / "frequent_chapter/with_dropped_sections",
]

for dir_path in train_data_dirs:
    print(f"\nDirectory: {dir_path.as_posix()}")
    
    if dir_path.exists() and dir_path.is_dir():
        parquet_files = list(dir_path.glob(f"*{extension}"))        
        if parquet_files:
            for file in parquet_files:
                print(f"{file.as_posix()}")
        else:
            print("No .parquet files found in this directory.")
    else:
        print(f"Directory not found: {dir_path}")

In [ ]:
dataset_path = Path("../data/processed/frequent_chapter/without_dropped_sections/modernbert_cleaned_t_s_dataset.parquet")

file_stem = Path(dataset_path).stem
sub_folders = Path(dataset_path).parent.relative_to(processed_data_dir)

fine_tuned_model_path = Path("../models/finetuned") / sub_folders / file_stem

print(f"Dataset path: {dataset_path.as_posix()}")
print(f"Model save path: {fine_tuned_model_path.as_posix()}")

### Adathalmaz betöltése és páciens-szintű, stratifikált felosztása tanító-, validációs- és teszthalmazra (Dataset loading and patient-level stratified split into train, validation, and test sets)

In [ ]:
df = pd.read_parquet(dataset_path, engine='pyarrow')
df_train, df_val, df_test, mlb, num_labels  = split_dataframe(df, "chapter", "subject_id", 10, SEED)

### Dataframe-ek átalakítása Dataset objektumokká (Converting DataFrames to Dataset objects)

#### Tokenizer betöltése és tokenizer függvény létrehozása (Loading tokenizer and creating tokenizer function)

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(local_model_path)

In [ ]:
MAX_LENGTH = 2048
TRUNCATION = True

def tokenize_function(texts):
    encoding = tokenizer(texts["text"], max_length=MAX_LENGTH, truncation=TRUNCATION)
    return encoding

In [ ]:
y_train = mlb.transform(df_train["chapter"]).astype("float32")
y_val   = mlb.transform(df_val["chapter"]).astype("float32")
y_test  = mlb.transform(df_test["chapter"]).astype("float32")

In [ ]:
dataset_train = Dataset.from_pandas(df_train)
dataset_val   = Dataset.from_pandas(df_val)
dataset_test  = Dataset.from_pandas(df_test)

In [ ]:
dataset_train = dataset_train.map(tokenize_function, batched=True)
dataset_val   = dataset_val.map(tokenize_function, batched=True)
dataset_test  = dataset_test.map(tokenize_function, batched=True)

In [ ]:
dataset_train = dataset_train.add_column("labels", y_train.tolist())
dataset_val   = dataset_val.add_column("labels", y_val.tolist())
dataset_test  = dataset_test.add_column("labels", y_test.tolist())

In [ ]:
cols_to_remove = list(df.columns)
cols_to_remove.append("__index_level_0__")
dataset_train = dataset_train.remove_columns([c for c in cols_to_remove if c in dataset_train.column_names])
dataset_val   = dataset_val.remove_columns([c for c in cols_to_remove if c in dataset_val.column_names])
dataset_test  = dataset_test.remove_columns([c for c in cols_to_remove if c in dataset_test.column_names])

In [ ]:
print("\nTrain dataset:")
print(dataset_train)
print("\nValidation dataset:")
print(dataset_val)
print("\nTest dataset:")
print(dataset_test)

### 


In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    local_model_path,
    num_labels=num_labels,
    problem_type="multi_label_classification",
    attn_implementation="flash_attention_2", 
    deterministic_flash_attn=True
)
model.to(device)

print(model)
print(type(model))
print(model.config.max_position_embeddings)

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    
    probs = expit(logits)
    
    preds = (probs >= 0.5).astype(int)

    micro_f1 = f1_score(y_true=labels, y_pred=preds, average="micro", zero_division=0)
    macro_f1 = f1_score(y_true=labels, y_pred=preds, average="macro", zero_division=0)
    weighted_f1 = f1_score(y_true=labels, y_pred=preds, average="weighted", zero_division=0)
    samples_f1 = f1_score(y_true=labels, y_pred=preds, average="samples", zero_division=0)

    micro_precision = precision_score(y_true=labels, y_pred=preds, average="micro", zero_division=0)
    macro_precision = precision_score(y_true=labels, y_pred=preds, average="macro", zero_division=0)

    micro_recall = recall_score(y_true=labels, y_pred=preds, average="micro", zero_division=0)
    macro_recall = recall_score(y_true=labels, y_pred=preds, average="macro", zero_division=0)

    hamming = hamming_loss(y_true=labels, y_pred=preds)
    
    accuracy = accuracy_score(y_true=labels, y_pred=preds)

    try:
        micro_auc = roc_auc_score(y_true=labels, y_score=probs, average="micro")
        macro_auc = roc_auc_score(y_true=labels, y_score=probs, average="macro")
    except ValueError :
        micro_auc = 0.0
        macro_auc = 0.0
 
    return {
        "accuracy": accuracy,
        "micro_f1": micro_f1,
        "macro_f1": macro_f1,
        "weighted_f1": weighted_f1,
        "samples_f1": samples_f1,
        "micro_precision": micro_precision,
        "macro_precision": macro_precision,
        "micro_recall": micro_recall,
        "macro_recall": macro_recall,
        "micro_auc": micro_auc,
        "macro_auc": macro_auc,
        "hamming_loss": hamming,
    }

In [ ]:
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

In [ ]:
training_args = TrainingArguments(
    output_dir=str(fine_tuned_model_path / "results"),
    logging_dir=str(fine_tuned_model_path / "logs"),
    report_to="tensorboard",
    eval_steps=500,
    save_steps=500,
    learning_rate=2e-5,             
    per_device_train_batch_size=2,  
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=16,
    num_train_epochs=4,              
    weight_decay=0.01,              
    save_total_limit=2,             
    load_best_model_at_end=True,     
    logging_strategy="steps", 
    logging_steps=50,              
    fp16=True,     
    eval_strategy="steps",
    save_strategy="steps",
    metric_for_best_model="micro_f1",     
    greater_is_better=True, 
    lr_scheduler_type="cosine",
    warmup_steps=500,
    seed=SEED,
)

In [ ]:
pos_counts = y_train.sum(axis=0)
neg_counts = len(y_train) - pos_counts
weights_v = torch.sqrt(torch.tensor(neg_counts / (pos_counts + 1e-8), dtype=torch.float32))

weights_df = pd.DataFrame({
    "chapter": mlb.classes_,
    "pos_count": pos_counts.astype(int),
    "weight": weights_v.numpy()
})

weights_df = weights_df.sort_values(by="weight", ascending=False)

print(weights_df.to_string(index=False))

pos_weight = weights_v.to(model.device)

In [ ]:
class CustomTrainer(Trainer):
    def __init__(self, pos_weight, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.pos_weight = pos_weight
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.get("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")
        loss_fct = nn.BCEWithLogitsLoss(pos_weight=self.pos_weight)
        loss = loss_fct(logits, labels)
        return (loss, outputs) if return_outputs else loss

In [ ]:
trainer = CustomTrainer(
    pos_weight=pos_weight,
    model=model,
    args=training_args,
    train_dataset=dataset_train,
    eval_dataset=dataset_val,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=4)]
)

In [ ]:
checkpoint_exists = False
if os.path.exists(training_args.output_dir):
    checkpoint_exists = any("checkpoint-" in d for d in os.listdir(training_args.output_dir))

trainer.train(resume_from_checkpoint=checkpoint_exists)

In [ ]:
best_model_path = fine_tuned_model_path / "best_model"
trainer.save_model(best_model_path)
tokenizer.save_pretrained(best_model_path)
mlb_path = best_model_path / "mlb.joblib"
joblib.dump(mlb, mlb_path)

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(best_model_path)

In [ ]:
eval_trainer = Trainer(
    model=model,
    args=training_args,
    compute_metrics=compute_metrics,
    data_collator=data_collator,
    eval_dataset = dataset_test
)

In [ ]:
results = eval_trainer.predict(dataset_test)

In [ ]:
logits = results.predictions
labels = results.label_ids

probs = expit(logits)
preds = (probs >= 0.5).astype(int)

report = classification_report(
    labels, 
    preds, 
    target_names=mlb.classes_, 
    zero_division=0,
    output_dict=True
)

report = pd.DataFrame(report).transpose()
summary_metrics = ["micro avg", "macro avg", "weighted avg", "samples avg"]

main_report = report.drop(index=[i for i in summary_metrics if i in report.index])
summary_report = report.loc[[i for i in summary_metrics if i in report.index]]

main_report = main_report.sort_values(by="f1-score", ascending=False)

report = pd.concat([main_report, summary_report])

print(json.dumps(results.metrics, indent=1))
print(report)

In [ ]:
eval_dir = fine_tuned_model_path / "evaluation_results"
eval_dir.mkdir(parents=True, exist_ok=True)

pd.DataFrame([results.metrics]).to_json(eval_dir / "metrics.json", orient='records', indent=1)

report.to_json(eval_dir / "classification_report.json", indent=1)

## ModernBERT finomhangolása top 50 ICD-10-CM kód osztályozásra (Fine-tuning ModernBERT for top 50 ICD-10-CM code classification) 

In [ ]:
processed_data_dir = Path("../data/processed")
extension = Path(".parquet")

train_data_dirs = [
    processed_data_dir / "top_50_code/without_dropped_sections",
    processed_data_dir / "top_50_code/with_dropped_sections",
]

for dir_path in train_data_dirs:
    print(f"\nDirectory: {dir_path.as_posix()}")

    if dir_path.exists() and dir_path.is_dir():
        parquet_files = list(dir_path.glob(f"*{extension}"))
        if parquet_files:
            for file in parquet_files:
                print(f"{file.as_posix()}")
        else:
            print("No .parquet files found in this directory.")
    else:
        print(f"Directory not found: {dir_path}")

In [ ]:
dataset_path = Path(
    "../data/processed/top_50_code/without_dropped_sections/modernbert_cleaned_t_s_dataset.parquet")

file_stem = Path(dataset_path).stem
sub_folders = Path(dataset_path).parent.relative_to(processed_data_dir)

fine_tuned_model_path = Path("../models/finetuned") / sub_folders / file_stem

print(f"Dataset path: {dataset_path.as_posix()}")
print(f"Model save path: {fine_tuned_model_path.as_posix()}")

### Adathalmaz betöltése és páciens-szintű, stratifikált felosztása tanító-, validációs- és teszthalmazra (Dataset loading and patient-level stratified split into train, validation, and test sets)

In [ ]:
df = pd.read_parquet(dataset_path, engine='pyarrow')
df_train, df_val, df_test, mlb, num_labels = split_dataframe(df, "icd_code", "subject_id", 10, SEED)

### Dataframe-ek átalakítása Dataset objektumokká (Converting DataFrames to Dataset objects)

#### Tokenizer betöltése és tokenizer függvény létrehozása (Loading tokenizer and creating tokenizer function)

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(local_model_path)

In [ ]:
MAX_LENGTH = 2048
TRUNCATION = True

def tokenize_function(texts):
    encoding = tokenizer(texts["text"], max_length=MAX_LENGTH, truncation=TRUNCATION)
    return encoding

In [ ]:
y_train = mlb.transform(df_train["icd_code"]).astype("float32")
y_val = mlb.transform(df_val["icd_code"]).astype("float32")
y_test = mlb.transform(df_test["icd_code"]).astype("float32")

In [ ]:
dataset_train = Dataset.from_pandas(df_train)
dataset_val = Dataset.from_pandas(df_val)
dataset_test = Dataset.from_pandas(df_test)

In [ ]:
dataset_train = dataset_train.map(tokenize_function, batched=True)
dataset_val = dataset_val.map(tokenize_function, batched=True)
dataset_test = dataset_test.map(tokenize_function, batched=True)

In [ ]:
dataset_train = dataset_train.add_column("labels", y_train.tolist())
dataset_val = dataset_val.add_column("labels", y_val.tolist())
dataset_test = dataset_test.add_column("labels", y_test.tolist())

In [ ]:
cols_to_remove = list(df.columns)
cols_to_remove.append("__index_level_0__")
dataset_train = dataset_train.remove_columns([c for c in cols_to_remove if c in dataset_train.column_names])
dataset_val = dataset_val.remove_columns([c for c in cols_to_remove if c in dataset_val.column_names])
dataset_test = dataset_test.remove_columns([c for c in cols_to_remove if c in dataset_test.column_names])
print("\nTrain dataset:")
print(dataset_train)
print("\nValidation dataset:")
print(dataset_val)
print("\nTest dataset:")
print(dataset_test)

### 

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    local_model_path,
    num_labels=num_labels,
    problem_type="multi_label_classification",
    attn_implementation="flash_attention_2",
    deterministic_flash_attn=True
)
model.to(device)

print(model)
print(type(model))
print(model.config.max_position_embeddings)

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred

    probs = expit(logits)

    preds = (probs >= 0.5).astype(int)

    micro_f1 = f1_score(y_true=labels, y_pred=preds, average="micro", zero_division=0)
    macro_f1 = f1_score(y_true=labels, y_pred=preds, average="macro", zero_division=0)
    weighted_f1 = f1_score(y_true=labels, y_pred=preds, average="weighted", zero_division=0)
    samples_f1 = f1_score(y_true=labels, y_pred=preds, average="samples", zero_division=0)

    micro_precision = precision_score(y_true=labels, y_pred=preds, average="micro", zero_division=0)
    macro_precision = precision_score(y_true=labels, y_pred=preds, average="macro", zero_division=0)

    micro_recall = recall_score(y_true=labels, y_pred=preds, average="micro", zero_division=0)
    macro_recall = recall_score(y_true=labels, y_pred=preds, average="macro", zero_division=0)

    hamming = hamming_loss(y_true=labels, y_pred=preds)

    accuracy = accuracy_score(y_true=labels, y_pred=preds)

    try:
        micro_auc = roc_auc_score(y_true=labels, y_score=probs, average="micro")
        macro_auc = roc_auc_score(y_true=labels, y_score=probs, average="macro")
    except ValueError:
        micro_auc = 0.0
        macro_auc = 0.0

    return {
        "accuracy": accuracy,
        "micro_f1": micro_f1,
        "macro_f1": macro_f1,
        "weighted_f1": weighted_f1,
        "samples_f1": samples_f1,
        "micro_precision": micro_precision,
        "macro_precision": macro_precision,
        "micro_recall": micro_recall,
        "macro_recall": macro_recall,
        "micro_auc": micro_auc,
        "macro_auc": macro_auc,
        "hamming_loss": hamming,
    }

In [ ]:
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

In [ ]:
training_args = TrainingArguments(
    output_dir=str(fine_tuned_model_path / "results"),
    logging_dir=str(fine_tuned_model_path / "logs"),
    report_to="tensorboard",
    eval_steps=500,
    save_steps=500,
    learning_rate=2e-5,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=16,
    num_train_epochs=4,
    weight_decay=0.01,
    save_total_limit=2,
    load_best_model_at_end=True,
    logging_strategy="steps",
    logging_steps=50,
    fp16=True,
    eval_strategy="steps",
    save_strategy="steps",
    metric_for_best_model="micro_f1",
    greater_is_better=True,
    lr_scheduler_type="cosine",
    warmup_steps=500,
    seed=SEED,
)

In [ ]:
pos_counts = y_train.sum(axis=0)
neg_counts = len(y_train) - pos_counts
weights_v = torch.sqrt(torch.tensor(neg_counts / (pos_counts + 1e-8), dtype=torch.float32))

weights_df = pd.DataFrame({
    "icd_code": mlb.classes_,
    "pos_count": pos_counts.astype(int),
    "weight": weights_v.numpy()
})

weights_df = weights_df.sort_values(by="weight", ascending=False)

print(weights_df.to_string(index=False))

pos_weight = weights_v.to(model.device)

In [ ]:
class CustomTrainer(Trainer):
    def __init__(self, pos_weight, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.pos_weight = pos_weight

    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.get("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")
        loss_fct = nn.BCEWithLogitsLoss(pos_weight=self.pos_weight)
        loss = loss_fct(logits, labels)
        return (loss, outputs) if return_outputs else loss

In [ ]:
trainer = CustomTrainer(
    pos_weight=pos_weight,
    model=model,
    args=training_args,
    train_dataset=dataset_train,
    eval_dataset=dataset_val,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=4)]
)

In [ ]:
checkpoint_exists = False
if os.path.exists(training_args.output_dir):
    checkpoint_exists = any("checkpoint-" in d for d in os.listdir(training_args.output_dir))

trainer.train(resume_from_checkpoint=checkpoint_exists)

In [ ]:
best_model_path = fine_tuned_model_path / "best_model"
trainer.save_model(best_model_path)
tokenizer.save_pretrained(best_model_path)
mlb_path = best_model_path / "mlb.joblib"
joblib.dump(mlb, mlb_path)

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(best_model_path)

In [ ]:
eval_trainer = Trainer(
    model=model,
    args=training_args,
    compute_metrics=compute_metrics,
    data_collator=data_collator,
    eval_dataset=dataset_test
)

In [ ]:
results = eval_trainer.predict(dataset_test)

In [ ]:
logits = results.predictions
labels = results.label_ids

probs = expit(logits)
preds = (probs >= 0.5).astype(int)

report = classification_report(
    labels,
    preds,
    target_names=mlb.classes_,
    zero_division=0,
    output_dict=True
)

report = pd.DataFrame(report).transpose()
summary_metrics = ["micro avg", "macro avg", "weighted avg", "samples avg"]

main_report = report.drop(index=[i for i in summary_metrics if i in report.index])
summary_report = report.loc[[i for i in summary_metrics if i in report.index]]

main_report = main_report.sort_values(by="f1-score", ascending=False)

report = pd.concat([main_report, summary_report])

print(json.dumps(results.metrics, indent=1))
print(report)

In [ ]:
eval_dir = fine_tuned_model_path / "evaluation_results"
eval_dir.mkdir(parents=True, exist_ok=True)

pd.DataFrame([results.metrics]).to_json(eval_dir / "metrics.json", orient='records', indent=1)

report.to_json(eval_dir / "classification_report.json", indent=1)